# Imports

In [ ]:
# Imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import langid

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import Dataset

In [ ]:
pd.set_option('max_colwidth', None)

# Load Data

In [ ]:
df = pd.read_csv(filepath_or_buffer='data/dataset.csv')
df.shape

In [ ]:
df.head()

In [ ]:
df['sentimento'].value_counts()

# Language Detection

In [ ]:
df['lang'] = None

In [ ]:
df

In [ ]:
# df.iterrows returns a tuple for each row (row_label, Series)

for index, row, in df.iterrows():
    
    text = row['texto']
    
    if isinstance(text, str):
        
        language, confidence = langid.classify(text)
        
        df.at[index, 'lang'] = language

In [ ]:
df['lang'].value_counts()

In [ ]:
df['lang'] = df['lang'].map({'gl': 'pt',
                             'fr': 'pt',
                             'pt': 'pt'})

In [ ]:
df['lang'].value_counts()

# Pre-Processing

## Data Cleaning

In [ ]:
df['texto'] = (
        df['texto']
        .str.replace(pat = r'http\S+', repl = "", regex = True)
        .str.replace(pat = r'#\w+', repl = "", regex = True)
        .str.replace(pat = r'@\w+', repl = "", regex = True)
        .str.replace(pat = r'\d+', repl = "", regex = True)
        .str.strip()
)

In [ ]:
df['texto']

## Label Encoder

In [ ]:
sentimento_labels = {'negativo': 0,
                     'neutro': 1,
                     'positivo': 2}

In [ ]:
df['labels'] = df['sentimento'].map(sentimento_labels)

In [ ]:
df.sample(10)

In [ ]:
n_labels = len(df['labels'].value_counts())

## Create Dataset object

In [ ]:
dataset = Dataset.from_pandas(df[['texto', 'labels']])
dataset

In [ ]:
dataset[20]

## Tokenization

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
#bert_tokenizer = BertTokenizer.from_pretrained('./model')

In [ ]:
def tokeniza_dados(text, tokenizer_obj, return_tensors = None):
    
    if isinstance(text, str):
        return tokenizer_obj(text, return_tensors = return_tensors, padding = False, truncation = True)
    
    if isinstance(text, list):
        return tokenizer_obj(text, return_tensors = return_tensors, padding = False, truncation = True)
    
    if isinstance(text, Dataset):
        return text.map(lambda x: tokenizer_obj(x['texto'],
                                                return_tensors = return_tensors,
                                                padding = False,
                                                truncation = True),
                        batched = True)

In [ ]:
# Teste com uma str
tokeniza_dados('hoje esá chovendo', bert_tokenizer)

In [ ]:
dataset_tokenizado = tokeniza_dados(dataset, bert_tokenizer)

In [ ]:
dataset_tokenizado

In [ ]:
dataset_tokenizado[1]['input_ids']

In [ ]:
# tokenizador do transfomers retorna uma instância da classe BatchEncoding

type(bert_tokenizer('hoje faz sol', return_tensors = 'pt', padding = False, truncation=True))

In [ ]:
a, b , c = bert_tokenizer('hoje faz sol', return_tensors = 'pt', padding = False, truncation=True).values()
a

## Split Data

In [ ]:
split = dataset_tokenizado.train_test_split(test_size=0.2)
split

In [ ]:
dataset_treino = split['train']
dataset_teste = split['test']

# Evaluation Metrics

In [ ]:
#from transformers import EvalPrediction
#help(EvalPrediction)

The Trainer object, used to train and evaluate the model, accepts the `compute_metrics` argument as a parameter.

`compute_metrics` must be a callable that takes an instance of the `EvalPrediction` class as input and returns a dictionary containing the calculated metrics.

`EvalPrediction` is a data structure (a `NamedTuple`) containing the following attributes:

.predictions: model outputs during evaluation (usually logits; this can be an `ndarray` or a tuple of arrays, depending on the task);

.label_ids: ground-truth labels corresponding to the evaluated samples.

The function passed to `compute_metrics` must process these values ​​(for example, by applying `argmax` to the logits) and return a dictionary.

In [ ]:
def calcula_metricas(evalprediction):
    
    logits, labels = evalprediction
    preds = np.argmax(logits, axis=1)
    
    return {'accuracy': accuracy_score(preds, labels)}

# Bert Model

https://huggingface.co/google-bert/bert-base-uncased

It is not exactly suitable for sentiment classification.

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = n_labels
)

When loading the `bert-base-uncased` pre-trained model, Hugging Face Transformers loads the weights for the BERT model layers that were trained during a pre-training task (e.g., masked word prediction).

However, the specific classification layers (such as `classifier.bias` and `classifier.weight`) are not loaded from the pre-trained model checkpoint. This is because these layers did not exist in the original pre-training model.

These layers are "new," created for the specific classification task you are defining (`BertForSequenceClassification` with `num_labels = 3`). Therefore, the weights for these layers are randomly initialized.

You must train the model on a **specific task ("downstream task")** using your own data.

## Training Hyperparameters

In [ ]:
training_args = TrainingArguments(
                            output_dir = './results',
                            overwrite_output_dir = True,
                        
                            eval_strategy = 'epoch',
                            save_strategy = 'best',
                            load_best_model_at_end = True,
                                                            
                            metric_for_best_model = 'accuracy',
                            greater_is_better = True,
                                
                            learning_rate = 2e-5,
                            per_device_train_batch_size = 8,
                            per_device_eval_batch_size = 8,
                            num_train_epochs = 10,
                            weight_decay = 0.01,
                            gradient_accumulation_steps = 1,
                            logging_steps = 2,
                            dataloader_drop_last=True

)

## Padding

In [ ]:
datacollator_obj = DataCollatorWithPadding(tokenizer = bert_tokenizer)

## Trainer

In [ ]:
import inspect
sig = inspect.signature(bert_model.forward).parameters
sig

In [ ]:
dataset_treino.features

In [ ]:
columns = []
for col in dataset_treino.features:
    for name in sig:
        if col == name:
           columns.append(col)
columns

In [ ]:
trainer_obj = Trainer(
    model = bert_model,
    args = training_args,
    data_collator = datacollator_obj,
    train_dataset = dataset_treino,
    eval_dataset = dataset_teste,
    processing_class = bert_tokenizer,
    compute_metrics = calcula_metricas)

In [ ]:
trainer_obj.train()

In [ ]:
#trainer_obj

In [ ]:
bert_tokenizer.save_pretrained('./model')
bert_model.save_pretrained('./model')

# Inference

## Line by line

In [ ]:
sentimento_labels

In [ ]:
labels = list(sentimento_labels.keys())
labels

In [ ]:
sample_text_1 = 'O Bitcoin é uma fraude, é um esquema Ponzi'

In [ ]:
inputs = tokeniza_dados(sample_text_1, bert_tokenizer, return_tensors= 'pt')

In [ ]:
inputs

In [ ]:
outputs = bert_model(**inputs)
outputs

In [ ]:
with torch.no_grad():
    logits = bert_model(**inputs).logits[0]
logits

In [ ]:
probs = F.softmax(input = logits, dim=-1)
probs

In [ ]:
predicted_class = torch.argmax(a = probs).item()
predicted_class

In [ ]:
predicted_label = labels[predicted_class]
predicted_label

In [ ]:
prob = probs[predicted_class].item()*100
prob

In [ ]:
f'A probabilidade do texto em questão ser classificado como {predicted_label} é de {probs[predicted_class].item()*100:.2f}%'

## Function

In [ ]:
def inference(texto, model):
    inputs = bert_tokenizer(texto, return_tensors = 'pt', padding = False, truncation=True)
    
    model.eval()
    
    with torch.inference_mode():
        outputs = model(**inputs)
        
    logits = outputs.logits
    
    probs = F.softmax(input = logits, dim = -1)
    
    predicted_class = torch.argmax(a = probs, dim = 1).item()
    predicted_label = labels[predicted_class]
    
    prob = probs[0, predicted_class].item() * 100
    
    return (f"A probabilidade do texto em questão ser classificado como {predicted_label}"
            f" é de {prob:.2f}%.")

In [ ]:
sample_text_2 = 'Bitcoin has no intrinsic value and should go to zero.'
inference(sample_text_2, bert_model)

# End